In [1]:
import gc
import logging
import os
import random
import shutil
import sys
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import requests
import yaml
from PIL import Image

In [2]:
try:
    import dotenv
    dotenv.load_dotenv(".env")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(".env")

# Настройка логгера
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("pipeline.log", encoding="utf-8", mode="a"),
    ],
)
log = logging.getLogger("pipeline")

try:
    import torch
    if torch.cuda.is_available():
        dev = torch.cuda.get_device_properties(0)
        vram_gb = dev.total_memory / 1024 ** 3
        log.info(f"GPU : {dev.name}")
        log.info(f"VRAM: {vram_gb:.1f} GB  |  CUDA {torch.version.cuda}")
        if vram_gb < 4:
            log.warning("VRAM < 4 GB — автоматический OOM recovery может не помочь")
    else:
        log.warning("CUDA не обнаружен — обучение на CPU (ОЧЕНЬ медленно)")
except ImportError:
    log.error("PyTorch не установлен!")

log.info(f"Python {sys.version.split()[0]} | cwd: {Path.cwd()}")


10:41:54 [INFO] GPU : NVIDIA GeForce RTX 2060
10:41:54 [INFO] VRAM: 6.0 GB  |  CUDA 12.4
10:41:54 [INFO] Python 3.13.1 | cwd: d:\MyProgramms\image-detection\pz-6


In [3]:
random.seed(42)

# Глобальные классы (порядок фиксирован)
GLOBAL_CLASSES = [
    "explicit_content",   # 0
    "weapon",             # 1
    "smoking",            # 2
    "alcohol",            # 3
    "drugs",              # 4
    "forbidden_symbols",  # 5
]
CLASS_TO_ID = {name: i for i, name in enumerate(GLOBAL_CLASSES)}

# Пути
BASE_DIR  = Path(".")
RAW_DIR   = BASE_DIR / "datasets_new" / "raw"     # сюда скачиваем
WORK_DIR  = BASE_DIR / "datasets_new" / "merged"  # сюда объединяем

SAMPLE_SIZE = 3000  # макс. изображений с одного источника на split

# API
RF_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
if RF_KEY:
    log.info(f"ROBOFLOW_API_KEY: найден ({RF_KEY[:8]}...)")
else:
    log.error("ROBOFLOW_API_KEY не найден — скачивание невозможно")

# Конфиг датасетов
#
# default_class : если задан — ВСЕ классы датасета, которых нет в class_mapping,
#                 маппятся в этот глобальный класс (удобно для однотемных датасетов)
# class_mapping : явный маппинг {исходный_класс -> глобальный или None (skip)}
#                 приоритет выше default_class
#
DATASETS_CONFIG = [

    # explicit_content
    {
        "name":          "adult_body",
        "workspace":     "obscenity",
        "project":       "adult-content-8gj6y",
        "version":       1,
        "default_class": "explicit_content",
        "class_mapping": {},
    },
    {
        "name":          "nudity_filter",
        "workspace":     "nsfwdetector",
        "project":       "nsfw-wh3c2",
        "version":       1,
        "default_class": "explicit_content",
        "class_mapping": {},
    },

    # weapon
    {
        "name":          "weapons_large",
        "workspace":     "yolov7test-u13vc",
        "project":       "weapon-detection-m7qso",
        "version":       16,
        "default_class": "weapon",
        "class_mapping": {},
    },

    # smoking
    {
        "name":          "smoking",
        "workspace":     "kyunghee-university-ada5d",
        "project":       "smoking-detection-3gefl",
        "version":       4,
        "default_class": "smoking",
        "class_mapping": {},
    },

    # alcohol
    {
        "name":          "alcohol",
        "workspace":     "adonantonin",
        "project":       "alcohol-iaeeq",
        "version":       4,
        "default_class": "alcohol",
        "class_mapping": {},
    },

    # drugs
    {
        "name":          "drugs_substances",
        "workspace":     "huangzixuan",
        "project":       "drugs-detection",
        "version":       1,
        "default_class": "drugs",
        "class_mapping": {},
    },
    {
        "name":          "drugs_syringe",
        "workspace":     "syringe-detection-nokm3",
        "project":       "syringe-detection-jbtuo",
        "version":       1,
        "default_class": "drugs",
        "class_mapping": {},
    },

    # forbidden_symbols
    {
        "name":          "hate_symbols",
        "workspace":     "computer-vision-y8bhq",
        "project":       "no-hate-symbols",
        "version":       1,
        "default_class": "forbidden_symbols",
        "class_mapping": {},
    },
]

log.info(f"Конфиг: {len(DATASETS_CONFIG)} источников → {len(GLOBAL_CLASSES)} классов")
for ds in DATASETS_CONFIG:
    log.info(f"  {ds['name']:20s}  ({ds['workspace']}/{ds['project']} v{ds['version']})  → {ds['default_class']}")


10:41:54 [INFO] ROBOFLOW_API_KEY: найден (mrDMkQZZ...)
10:41:54 [INFO] Конфиг: 8 источников → 6 классов
10:41:54 [INFO]   adult_body            (obscenity/adult-content-8gj6y v1)  → explicit_content
10:41:54 [INFO]   nudity_filter         (nsfwdetector/nsfw-wh3c2 v1)  → explicit_content
10:41:54 [INFO]   weapons_large         (yolov7test-u13vc/weapon-detection-m7qso v16)  → weapon
10:41:54 [INFO]   smoking               (kyunghee-university-ada5d/smoking-detection-3gefl v4)  → smoking
10:41:54 [INFO]   alcohol               (adonantonin/alcohol-iaeeq v4)  → alcohol
10:41:54 [INFO]   drugs_substances      (huangzixuan/drugs-detection v1)  → drugs
10:41:54 [INFO]   drugs_syringe         (syringe-detection-nokm3/syringe-detection-jbtuo v1)  → drugs
10:41:54 [INFO]   hate_symbols          (computer-vision-y8bhq/no-hate-symbols v1)  → forbidden_symbols


In [4]:
def download_roboflow(
    workspace: str,
    project: str,
    version: int,
    api_key: str,
    out_dir: Path,
    fallback_versions: list[int] | None = None,
) -> Path:
    if (out_dir / "data.yaml").exists():
        log.info(f"  [{project}] Уже скачан → {out_dir}")
        return out_dir

    if not api_key:
        raise RuntimeError("ROBOFLOW_API_KEY не задан")

    versions_to_try = [version] + (fallback_versions or [])

    for ver in versions_to_try:
        url = f"https://api.roboflow.com/{workspace}/{project}/{ver}/yolov8"
        log.info(f"  [{project}] Запрос v{ver}...")
        try:
            resp = requests.get(url, params={"api_key": api_key}, timeout=30)
        except requests.RequestException as e:
            log.warning(f"  [{project}] v{ver} сетевая ошибка: {e}")
            continue

        if resp.status_code != 200:
            log.warning(f"  [{project}] v{ver} HTTP {resp.status_code} — пробуем следующую версию")
            continue

        link = resp.json().get("export", {}).get("link")
        if not link:
            log.warning(f"  [{project}] v{ver} нет ссылки: {resp.json()}")
            continue

        zip_path = out_dir.parent / f"_tmp_{project}_v{ver}.zip"
        log.info(f"  [{project}] Скачивание v{ver}...")
        try:
            with requests.get(link, stream=True, timeout=300) as r:
                r.raise_for_status()
                total = int(r.headers.get("content-length", 0))
                done = 0
                with open(zip_path, "wb") as f:
                    for chunk in r.iter_content(8192):
                        f.write(chunk)
                        done += len(chunk)
                        if total:
                            print(f"\r  {done / total * 100:.1f}% ({done // 1024 // 1024} MB)",
                                  end="", flush=True)
            print()
        except Exception as e:
            log.warning(f"  [{project}] v{ver} ошибка скачивания: {e}")
            zip_path.unlink(missing_ok=True)
            continue

        out_dir.mkdir(parents=True, exist_ok=True)
        log.info(f"  [{project}] Распаковка...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(out_dir)
        zip_path.unlink(missing_ok=True)
        log.info(f"  [{project}] v{ver} ✓")
        return out_dir

    raise RuntimeError(
        f"Не удалось скачать {project} ни для одной из версий: {versions_to_try}"
    )


# ─── Execute downloads ────────────────────────────────────────────────────────
RAW_DIR.mkdir(parents=True, exist_ok=True)
download_errors: list[str] = []

for ds in DATASETS_CONFIG:
    out = RAW_DIR / ds["name"]
    log.info(f"\n[Download] {ds['name']}  ({ds['workspace']}/{ds['project']})")
    try:
        download_roboflow(
            ds["workspace"], ds["project"], ds["version"], RF_KEY, out,
            fallback_versions=[1, 2, 3],  # пробуем ранние версии если latest недоступна
        )
    except Exception as exc:
        log.error(f"  ОШИБКА: {exc}")
        download_errors.append(ds["name"])

if download_errors:
    log.warning(f"\nПропущены: {download_errors}")
    log.warning("Проверьте ROBOFLOW_API_KEY и названия проектов.")
else:
    log.info("\nВсе датасеты скачаны ✓")


10:41:55 [INFO] 
[Download] adult_body  (obscenity/adult-content-8gj6y)
10:41:55 [INFO]   [adult-content-8gj6y] Уже скачан → datasets_new\raw\adult_body
10:41:55 [INFO] 
[Download] nudity_filter  (nsfwdetector/nsfw-wh3c2)
10:41:55 [INFO]   [nsfw-wh3c2] Уже скачан → datasets_new\raw\nudity_filter
10:41:55 [INFO] 
[Download] weapons_large  (yolov7test-u13vc/weapon-detection-m7qso)
10:41:55 [INFO]   [weapon-detection-m7qso] Уже скачан → datasets_new\raw\weapons_large
10:41:55 [INFO] 
[Download] smoking  (kyunghee-university-ada5d/smoking-detection-3gefl)
10:41:55 [INFO]   [smoking-detection-3gefl] Уже скачан → datasets_new\raw\smoking
10:41:55 [INFO] 
[Download] alcohol  (adonantonin/alcohol-iaeeq)
10:41:55 [INFO]   [alcohol-iaeeq] Уже скачан → datasets_new\raw\alcohol
10:41:55 [INFO] 
[Download] drugs_substances  (huangzixuan/drugs-detection)
10:41:55 [INFO]   [drugs-detection] Уже скачан → datasets_new\raw\drugs_substances
10:41:55 [INFO] 
[Download] drugs_syringe  (syringe-detection-no

In [5]:
# Smart class mapper

def load_yaml_classes(ds_dir: Path) -> list[str]:
    yaml_path = ds_dir / "data.yaml"
    if not yaml_path.exists():
        found = list(ds_dir.rglob("data.yaml"))
        if not found:
            raise FileNotFoundError(f"data.yaml не найден в {ds_dir}")
        yaml_path = sorted(found)[0]
    with open(yaml_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    names = cfg.get("names", [])
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names.keys())]
    return names


def build_id_remap(
    src_classes: list[str],
    class_mapping: dict[str, str | None],
    default_class: str | None,
    ds_name: str = "",
) -> dict[int, int | None]:
    """
    Builds {src_id -> global_id | None}.

    Priority:
      1. explicit class_mapping  (exact, then case-insensitive)
      2. default_class           (if set, catches anything not in class_mapping)
      3. None / skip             (warning)
    """
    remap: dict[int, int | None] = {}

    for src_id, src_name in enumerate(src_classes):
        # 1. Exact match in class_mapping
        if src_name in class_mapping:
            target = class_mapping[src_name]
        else:
            # Case-insensitive lookup in class_mapping
            target = next(
                (v for k, v in class_mapping.items() if k.lower() == src_name.lower()),
                "__not_found__",
            )

        if target == "__not_found__":
            # 2. Fall back to default_class
            if default_class is not None:
                if default_class not in CLASS_TO_ID:
                    log.error(f"  [{ds_name}] default_class '{default_class}' не является глобальным!")
                    remap[src_id] = None
                else:
                    gid = CLASS_TO_ID[default_class]
                    log.info(f"  [{ds_name}]   '{src_name}' (src={src_id}) → '{default_class}' (default, gid={gid})")
                    remap[src_id] = gid
            else:
                # 3. No mapping found
                log.warning(f"  [{ds_name}] ⚠ ПРОПУСК: '{src_name}' (src={src_id}) — нет в маппинге")
                remap[src_id] = None

        elif target is None:
            log.info(f"  [{ds_name}]   '{src_name}' (src={src_id}) → ПРОПУСК (явный None)")
            remap[src_id] = None

        elif target not in CLASS_TO_ID:
            log.error(f"  [{ds_name}] ✗ КОНФЛИКТ: '{target}' не глобальный класс — пропуск")
            remap[src_id] = None

        else:
            gid = CLASS_TO_ID[target]
            log.info(f"  [{ds_name}]   '{src_name}' (src={src_id}) → '{target}' (gid={gid})")
            remap[src_id] = gid

    mapped = sum(1 for v in remap.values() if v is not None)
    log.info(f"  [{ds_name}] Итог: {mapped}/{len(src_classes)} классов смаппировано")
    return remap


log.info("Smart mapper загружен ✓")


10:41:55 [INFO] Smart mapper загружен ✓


In [6]:
# ─── Validation + label remapping ────────────────────────────────────────────

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def is_valid_image(path: Path) -> bool:
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False


def find_split_dir(ds_dir: Path, split: str) -> Path | None:
    """Handles train/valid/val/test naming variations."""
    aliases = {
        "train": ["train"],
        "valid": ["valid", "val", "validation"],
        "test":  ["test"],
    }
    for alias in aliases.get(split, [split]):
        p = ds_dir / alias
        if (p / "images").exists():
            return p
    return None


def remap_label_file(
    src: Path,
    dst: Path,
    id_remap: dict[int, int | None],
) -> tuple[bool, int]:
    """Remaps class IDs, clips bboxes to [0,1]. Returns (has_boxes, count)."""
    if not src.exists():
        return False, 0
    try:
        raw = src.read_text(encoding="utf-8").strip()
    except Exception:
        return False, 0

    lines_out = []
    for line in raw.splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            src_cls = int(parts[0])
            xc, yc, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        except ValueError:
            continue

        dst_cls = id_remap.get(src_cls)
        if dst_cls is None:
            continue

        # Clip to image boundaries
        x1, y1 = max(0.0, xc - w / 2), max(0.0, yc - h / 2)
        x2, y2 = min(1.0, xc + w / 2), min(1.0, yc + h / 2)
        w_c, h_c = x2 - x1, y2 - y1
        if w_c < 1e-4 or h_c < 1e-4:
            continue

        lines_out.append(f"{dst_cls} {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {w_c:.6f} {h_c:.6f}")

    if lines_out:
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.write_text("\n".join(lines_out), encoding="utf-8")
        return True, len(lines_out)
    return False, 0


def merge_split(
    ds: dict,
    split: str,
    dst_dir: Path,
    id_remap: dict[int, int | None],
    class_counts: dict | None = None,
) -> dict:
    """Merges one dataset split into dst_dir."""
    src_split = find_split_dir(RAW_DIR / ds["name"], split)
    if src_split is None:
        log.warning(f"  [{ds['name']}/{split}] Директория не найдена")
        return {"copied": 0, "corrupt": 0, "empty": 0}

    src_imgs = src_split / "images"
    src_lbls = src_split / "labels"
    dst_imgs = (dst_dir / "images")
    dst_lbls = (dst_dir / "labels")
    dst_imgs.mkdir(parents=True, exist_ok=True)
    dst_lbls.mkdir(parents=True, exist_ok=True)

    all_imgs = [p for p in src_imgs.iterdir() if p.suffix.lower() in IMG_EXTS]
    if len(all_imgs) > SAMPLE_SIZE:
        all_imgs = random.sample(all_imgs, SAMPLE_SIZE)

    stats = {"copied": 0, "corrupt": 0, "empty": 0}
    prefix = ds["name"]

    for img_path in all_imgs:
        if not is_valid_image(img_path):
            stats["corrupt"] += 1
            continue

        base     = img_path.stem
        dst_name = f"{prefix}_{base}"
        src_lbl  = src_lbls / f"{base}.txt"
        dst_img  = dst_imgs / f"{dst_name}{img_path.suffix}"
        dst_lbl  = dst_lbls / f"{dst_name}.txt"

        has_boxes, _ = remap_label_file(src_lbl, dst_lbl, id_remap)
        if not has_boxes:
            stats["empty"] += 1
            continue

        shutil.copy2(img_path, dst_img)
        stats["copied"] += 1

        if class_counts is not None:
            for line in dst_lbl.read_text(encoding="utf-8").splitlines():
                p = line.split()
                if p:
                    class_counts[int(p[0])] += 1

    return stats


log.info("Функции валидации загружены ✓")


10:41:59 [INFO] Функции валидации загружены ✓


In [7]:
data_yaml_path = WORK_DIR / "data.yaml"

In [ ]:

if data_yaml_path.exists():
    log.info(f"Merged датасет уже существует: {data_yaml_path}")
    log.info("Для пересборки удалите папку datasets_new/merged/")
else:
    log.info("=== Сборка объединённого датасета ===")
    WORK_DIR.mkdir(parents=True, exist_ok=True)

    train_counts: dict[int, int] = defaultdict(int)

    for ds in DATASETS_CONFIG:
        raw_path = RAW_DIR / ds["name"]
        if not raw_path.exists():
            log.warning(f"[{ds['name']}] Директория не найдена — пропуск")
            continue

        log.info(f"\n─── {ds['name']} ───")
        src_classes = load_yaml_classes(raw_path)
        log.info(f"  Исходные классы: {src_classes}")
        id_remap = build_id_remap(
            src_classes,
            ds["class_mapping"],
            ds.get("default_class"),
            ds["name"],
        )

        for split in ["train", "valid", "test"]:
            stats = merge_split(
                ds, split, WORK_DIR / split, id_remap,
                class_counts=train_counts if split == "train" else None,
            )
            log.info(
                f"  {split:5s}: скопировано={stats['copied']:4d}  "
                f"битых={stats['corrupt']:2d}  пустых={stats['empty']:4d}"
            )

    # ─── data.yaml ───────────────────────────────────────────────────────────
    cfg = {
        "path":  str(WORK_DIR.resolve()),
        "train": "train/images",
        "val":   "valid/images",
        "test":  "test/images",
        "nc":    len(GLOBAL_CLASSES),
        "names": GLOBAL_CLASSES,
    }
    with open(data_yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

    # ─── Статистика классов ───────────────────────────────────────────────────
    log.info("\n=== Распределение классов в train ===")
    max_cnt = max(train_counts.values(), default=1)
    for cls_id in range(len(GLOBAL_CLASSES)):
        cnt  = train_counts.get(cls_id, 0)
        bar  = "█" * int(cnt * 35 / max_cnt)
        flag = " ⚠ МАЛО (<200)" if cnt < 200 else ""
        log.info(f"  [{cls_id}] {GLOBAL_CLASSES[cls_id]:20s}: {cnt:5d}  {bar}{flag}")
    log.info(f"\n  Всего объектов (train): {sum(train_counts.values())}")
    log.info(f"  data.yaml сохранён: {data_yaml_path}")


10:44:22 [INFO] === Сборка объединённого датасета ===
10:44:22 [INFO] 
─── adult_body ───
10:44:22 [INFO]   Исходные классы: ['breast_nude', 'breasts_seminude', 'buttock', 'genitalia_female', 'genitalia_male']
10:44:22 [INFO]   [adult_body]   'breast_nude' (src=0) → 'explicit_content' (default, gid=0)
10:44:22 [INFO]   [adult_body]   'breasts_seminude' (src=1) → 'explicit_content' (default, gid=0)
10:44:22 [INFO]   [adult_body]   'buttock' (src=2) → 'explicit_content' (default, gid=0)
10:44:22 [INFO]   [adult_body]   'genitalia_female' (src=3) → 'explicit_content' (default, gid=0)
10:44:22 [INFO]   [adult_body]   'genitalia_male' (src=4) → 'explicit_content' (default, gid=0)
10:44:22 [INFO]   [adult_body] Итог: 5/5 классов смаппировано
10:44:40 [INFO]   train: скопировано= 864  битых= 0  пустых=   0
10:44:44 [INFO]   valid: скопировано= 247  битых= 0  пустых=   0
10:44:46 [INFO]   test : скопировано= 123  битых= 0  пустых=   0
10:44:46 [INFO] 
─── nudity_filter ───
10:44:46 [INFO]   Ис

In [ ]:
# ─── Dataset summary ─────────────────────────────────────────────────────────
print(f"{'Split':<8} {'Images':>8} {'Labels':>8}")
print("-" * 26)
for split in ["train", "valid", "test"]:
    imgs_dir = WORK_DIR / split / "images"
    lbls_dir = WORK_DIR / split / "labels"
    n_imgs = len(list(imgs_dir.glob("*"))) if imgs_dir.exists() else 0
    n_lbls = len(list(lbls_dir.glob("*.txt"))) if lbls_dir.exists() else 0
    print(f"{split:<8} {n_imgs:>8} {n_lbls:>8}")

print(f"\ndata.yaml: {data_yaml_path}")
print(f"Classes  : {GLOBAL_CLASSES}")


Split      Images   Labels
--------------------------
train       16221    16221
valid        4612     4612
test         1650     1650

data.yaml: datasets_new\merged\data.yaml
Classes  : ['explicit_content', 'weapon', 'smoking', 'alcohol', 'drugs', 'forbidden_symbols']


In [ ]:
# ─── OOM-safe training ────────────────────────────────────────────────────────

def train_with_oom_recovery(
    data_yaml: str | Path,
    model_weights: str = "yolov8s.pt",
    epochs: int = 50,
    init_batch: int = 64,
    init_imgsz: int = 640,
    project: str = "runs/train",
    run_name: str = "content_mod",
):
    """
    YOLOv8 training with automatic CUDA OOM recovery.

    Recovery strategy (RTX 2060 6 GB):
      batch: 8 → 4 → 2
      imgsz: 640 → 576 → 512 → ... → 320
    Each retry: gc.collect() + cuda.empty_cache() + 3s sleep.
    """
    import torch
    from ultralytics import YOLO

    batch   = init_batch
    imgsz   = init_imgsz
    attempt = 0
    max_attempts = 8

    while attempt < max_attempts:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        log.info("\n" + "=" * 60)
        log.info(f"Попытка {attempt + 1}/{max_attempts}  |  batch={batch}  imgsz={imgsz}")
        log.info("=" * 60)

        try:
            model = YOLO(model_weights)
            results = model.train(
                data=str(data_yaml),
                epochs=epochs,
                imgsz=imgsz,
                batch=batch,
                device=0 if torch.cuda.is_available() else "cpu",
                workers=4,
                # ── Экономия VRAM (RTX 2060 6 GB) ────────────────────────────
                amp=True,           # Mixed Precision — обязательно
                cache="disk",       # disk cache, не нагружает VRAM
                # ── Оптимизатор ───────────────────────────────────────────────
                optimizer="AdamW",
                lr0=0.001,
                lrf=0.01,
                weight_decay=0.0005,
                momentum=0.937,
                # ── Расписание ────────────────────────────────────────────────
                warmup_epochs=3,
                cos_lr=True,
                # ── Аугментация ───────────────────────────────────────────────
                close_mosaic=10,
                hsv_h=0.015,
                hsv_s=0.7,
                hsv_v=0.4,
                fliplr=0.5,
                flipud=0.0,
                translate=0.1,
                scale=0.5,
                mosaic=1.0,
                mixup=0.0,
                # ── Остановка / сохранение ────────────────────────────────────
                patience=10,
                save=True,
                save_period=10,
                project=project,
                name=f"{run_name}_b{batch}_sz{imgsz}",
                exist_ok=True,
                verbose=True,
            )
            log.info(f"\n✓ Обучение завершено! Результаты: {model.trainer.save_dir}")
            return model, results

        except Exception as exc:
            is_oom = isinstance(exc, torch.cuda.OutOfMemoryError) or (
                isinstance(exc, RuntimeError)
                and "out of memory" in str(exc).lower()
            )
            if not is_oom:
                log.error(f"Не-OOM ошибка: {exc}")
                raise

            attempt += 1
            log.warning(f"\nCUDA OOM! (попытка {attempt})")
            gc.collect()
            torch.cuda.empty_cache()
            time.sleep(3)

            if batch > 4:
                batch = max(2, batch // 2)
                log.info(f"  ↓ batch_size → {batch}")
            elif imgsz > 320:
                imgsz = max(320, imgsz - 64)
                batch = max(2, batch)
                log.info(f"  ↓ imgsz → {imgsz}  batch={batch}")
            else:
                log.error("Минимум достигнут (batch=2, imgsz=320). Освободите VRAM вручную.")
                raise

    raise RuntimeError(f"Обучение не удалось после {max_attempts} попыток OOM")


model, train_results = train_with_oom_recovery(
    data_yaml=data_yaml_path,
    model_weights="yolov8s.pt",
    epochs=50,
    init_batch=16,
    init_imgsz=640,
    project="runs/train",
    run_name="content_moderation",
)


20:07:09 [INFO] Датасет : datasets_new\merged\data.yaml
20:07:09 [INFO] Модель  : yolov8s.pt (COCO pretrained)
20:07:09 [INFO] Железо  : RTX 2060 6 GB | AMP + disk cache
20:07:09 [INFO] 
20:07:09 [INFO] Попытка 1/8  |  batch=16  imgsz=640
20:07:09 [INFO] ============================================================
New https://pypi.org/project/ultralytics/8.4.51 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.45  Python-3.13.1 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=datasets_new\merged\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.

### Дообучение на новом датасете с наркотиками

In [ ]:
import shutil
from pathlib import Path

import yaml
from ultralytics import YOLO


In [17]:
# Создаём data.yaml только с drugs датасетом
drugs_yaml = {
    "path":  str((WORK_DIR / "drugs_only").resolve()),
    "train": "train/images",
    "val":   "valid/images",
    "nc":    6,
    "names": GLOBAL_CLASSES,  # все 6 классов — модель должна знать их все
}
drugs_dir = WORK_DIR / "drugs_only"
for split in ["train/images", "train/labels", "valid/images", "valid/labels"]:
    (drugs_dir / split).mkdir(parents=True, exist_ok=True)

# Копируем только drugs-изображения из merged датасета

for split in ["train", "valid"]:
    src_imgs = WORK_DIR / split / "images"
    src_lbls = WORK_DIR / split / "labels"
    for lbl_path in src_lbls.glob("*.txt"):
        classes_in_file = {int(l.split()[0]) for l in lbl_path.read_text().splitlines() if l}
        if 4 in classes_in_file:  # 4 = drugs
            img = next(src_imgs.glob(f"{lbl_path.stem}.*"), None)
            if img:
                shutil.copy2(img, drugs_dir / split / "images" / img.name)
                shutil.copy2(lbl_path, drugs_dir / split / "labels" / lbl_path.name)

with open(drugs_dir / "data.yaml", "w") as f:
    yaml.dump(drugs_yaml, f, allow_unicode=True)

n_train = len(list((drugs_dir / "train/images").glob("*")))
n_val   = len(list((drugs_dir / "valid/images").glob("*")))
print(f"Drugs-only: train={n_train}, val={n_val}")


Drugs-only: train=1374, val=187


In [19]:

# Fine-tune
model = YOLO("best.pt")
model.train(
    data=str(drugs_dir / "data.yaml"),
    epochs=30,
    freeze=21,       # заморозить backbone (только detection head обучается)
    lr0=0.0001,      # в 10x меньше стандартного — защита от forgetting
    batch=16,
    imgsz=640,
    amp=True,
    cache="ram",
    optimizer="AdamW",
    patience=7,
    project="runs/train",
    name="drugs_finetune",
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.51 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.45  Python-3.13.1 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets_new\merged\drugs_only\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=21, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=best.pt, momentum=0.937, mosaic=1.0, multi_sc

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001550CDE7EE0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

### Итоговые результаты модели

In [20]:
import torch
from ultralytics import YOLO
from pathlib import Path


In [22]:
def evaluate_model(weights_path: str | Path, label: str) -> dict:
    model = YOLO(str(weights_path))
    metrics = model.val(
        data=str(data_yaml_path),
        split="test",
        imgsz=640,
        batch=8,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False,
    )

    results = {}
    if hasattr(metrics, "ap_class_index") and metrics.ap_class_index is not None:
        for i, cls_idx in enumerate(metrics.ap_class_index):
            name = GLOBAL_CLASSES[cls_idx] if cls_idx < len(GLOBAL_CLASSES) else f"cls_{cls_idx}"
            results[name] = {
                "P":       float(metrics.box.p[i]),
                "R":       float(metrics.box.r[i]),
                "mAP50":   float(metrics.box.ap50[i]),
                "mAP50-95":float(metrics.box.ap[i]),
            }

    results["__all__"] = {
        "P":        float(metrics.box.mp),
        "R":        float(metrics.box.mr),
        "mAP50":    float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
    }
    print(f"\n{'─'*65}  {label}")
    print(f"  {'Класс':<22} {'P':>7} {'R':>7} {'mAP50':>7} {'mAP50-95':>9}")
    print(f"  {'─'*58}")
    for name, m in results.items():
        tag = " ← drugs" if name == "drugs" else ""
        prefix = "  ALL" if name == "__all__" else f"  {name}"
        print(f"{prefix:<26} {m['P']:>7.3f} {m['R']:>7.3f} {m['mAP50']:>7.3f} {m['mAP50-95']:>9.3f}{tag}")
    return results


# Пути к моделям
original_weights  = Path("best.pt")
finetuned_weights = Path("D:\\MyProgramms\\image-detection\\runs\\detect\\runs\\train\\drugs_finetune\\weights\\best.pt")

scores = {}
if original_weights.exists():
    scores["original"] = evaluate_model(original_weights, "ОРИГИНАЛ")

if finetuned_weights.exists():
    scores["finetuned"] = evaluate_model(finetuned_weights, "ПОСЛЕ ДООБУЧЕНИЯ")
else:
    print(f"Дообученная модель не найдена: {finetuned_weights}")


# Дельта: что изменилось
if "original" in scores and "finetuned" in scores:
    print(f"\n{'─'*65}  ИЗМЕНЕНИЕ (finetuned − original)")
    print(f"  {'Класс':<22} {'ΔP':>7} {'ΔR':>7} {'ΔmAP50':>7} {'ΔmAP50-95':>9}")
    print(f"  {'─'*58}")
    for name in scores["original"]:
        o = scores["original"][name]
        f = scores["finetuned"].get(name, o)
        dm50 = f["mAP50"] - o["mAP50"]
        sign = lambda x: ("+" if x >= 0 else "") + f"{x:.3f}"
        flag = " ⚠" if name != "drugs" and dm50 < -0.02 else ""
        flag = " ✓" if name == "drugs" and dm50 > 0 else flag
        print(f"  {name:<24} {sign(f['P']-o['P']):>7} {sign(f['R']-o['R']):>7} "
              f"{sign(dm50):>7} {sign(f['mAP50-95']-o['mAP50-95']):>9}{flag}")

Ultralytics 8.4.45  Python-3.13.1 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 73 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 38.076.8 MB/s, size: 39.0 KB)
val: Scanning D:\MyProgramms\image-detection\pz-6\datasets_new\merged\test\labels.cache... 1650 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1650/1650 576.7Mit/s 0.0s
val: D:\MyProgramms\image-detection\pz-6\datasets_new\merged\test\images\adult_body_prefix_pornhub_127740_jpeg_jpg.rf.59e3bf241d32f402f118d973377de0eb.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 207/207 9.9it/s 21.0ss<0.1s
                   all       1650       3504      0.659       0.56      0.615       0.43
Speed: 1.1ms preprocess, 7.6ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to D:\MyProgramms\image-detection\runs\detect\val-4

───────────────